# LC 268 — Missing Number
**Difficulty:** Easy | **Category:** Bit Manipulation
**Pattern:** XOR Index-Value Cancellation (or Gauss Sum)

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> XOR all indices 0..n with all values
in nums. Every present number cancels its index. Only the missing
index has no value to cancel it — so it's what remains.
Alternatively: expected sum minus actual sum equals the gap.
</div>

## Official Problem Statement

Given an array `nums` containing `n` distinct numbers in the range
`[0, n]`, return the only number in the range that is missing
from the array.

**Constraints:**
- `n == nums.length`
- `1 <= n <= 10^4`
- `0 <= nums[i] <= n`
- All numbers in `nums` are unique

**Follow-up:** Can you solve it in O(1) extra space and O(n) time?

## What This Is Actually Asking

You have a list of n numbers that should contain every value
from 0 to n — but one is missing. Which one?

Think of it as a complete set of numbered tokens with one removed.
You need to find the gap without sorting or using extra space.

Two clean solutions exist: math (Gauss sum) or XOR cancellation.

## Walk Through an Example by Hand

Input: `nums = [3, 0, 1]`  →  n = 3, expected: `2`

**XOR Method:**
```
XOR indices 0..3 with all values:

result = 0
XOR index 0:  result = 0 ^ 0 = 0
XOR index 1:  result = 0 ^ 1 = 1
XOR index 2:  result = 1 ^ 2 = 3
XOR index 3:  result = 3 ^ 3 = 0   ← n itself
XOR value 3:  result = 0 ^ 3 = 3
XOR value 0:  result = 3 ^ 0 = 3
XOR value 1:  result = 3 ^ 1 = 2
```
**Answer: 2**  (index 2 had no matching value to cancel it)

**Gauss Sum Method:**
```
expected = 3*(3+1)//2 = 6
actual   = 3+0+1 = 4
missing  = 6 - 4 = 2  ✓
```

## The Picture

```
Complete set for n=3: {0, 1, 2, 3}
Given:               {0, 1,    3}  ← 2 is missing

XOR view — pair each index with each value:

  idx 0  ↔  val 0   →  0 ^ 0 = 0  ✓ cancelled
  idx 1  ↔  val 1   →  1 ^ 1 = 0  ✓ cancelled
  idx 2  ↔  ???     →  2 alone     ← survives!
  idx 3  ↔  val 3   →  3 ^ 3 = 0  ✓ cancelled

Gauss Sum view:

  Expected:  0+1+2+3 = 6  (triangle number: n*(n+1)/2)
  Actual:    0+1+3   = 4
  Gap:       6 - 4   = 2  ← missing number

Both methods: O(n) time, O(1) space.
Gauss Sum is simpler to read. XOR avoids overflow.
```

## When To Use This Pattern

- When you have a **nearly complete sequence** with one gap.
- When the problem is "find missing in range [0..n]".
- XOR approach: when you want **overflow safety** for large n.
- Gauss sum: when you want **simplest possible code**.
- Generalizes to: find missing in sorted rotated array
  (binary search variant).

## The Approach

**XOR method:** Initialize result to n. XOR every index i with
its value nums[i]. Every matched pair cancels. The missing
index survives as the result.

**Gauss method:** Compute expected = n*(n+1)//2. Subtract the
actual sum of nums. The difference is the missing number.
Both are O(n) time, O(1) space — choose whichever is clearest.

In [1]:
from typing import List  # standard collection types

In [2]:
def test_harness(func):
    tests = [
        # (input, expected, label)
        ([3, 0, 1],       2, "standard: missing 2"),
        ([0, 1],          2, "missing last: n=2"),
        ([9,6,4,2,3,5,7,0,1], 8, "large: missing 8"),
        ([0],             1, "edge: n=1, missing 1"),
        ([1],             0, "edge: n=1, missing 0"),
    ]
    passed = 0
    for nums, expected, label in tests:
        result = func(nums)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(f"  [{status}] {label}")
        if status == "FAILED":
            print(f"           got={result}, expected={expected}")
    print(f"\n  {passed}/{len(tests)} tests passed")

In [14]:
def missingNumber(nums: List[int]) -> int:
    """
    Find the missing number in range [0, n] given n distinct
    numbers from that range.

    Approach A (XOR): result = n, then for each i: result ^= i ^ nums[i]
      Paired values cancel; missing index survives.

    Approach B (Gauss): n*(n+1)//2 - sum(nums)
      Expected sum minus actual sum = missing value.

    Time:  O(n)
    Space: O(1)
    """
    res = len(nums)
    for i, num in enumerate(nums):
        res = res ^ i ^ num
    return res

r'''
2
None
8
None
0

'''

# Debug prints — expected values shown in comments
print(missingNumber([3, 0, 1]))               # expected: 2
print(missingNumber([0, 1]))                  # expected: 2
print(missingNumber([9,6,4,2,3,5,7,0,1]))    # expected: 8
print(missingNumber([0]))                     # expected: 1
print(missingNumber([1]))                     # expected: 0
print(missingNumber([8,6,4,2,3,5,7,0,1]))    # expected: 9
test_harness(missingNumber)

2
2
8
1
0
9
  [PASSED] standard: missing 2
  [PASSED] missing last: n=2
  [PASSED] large: missing 8
  [PASSED] edge: n=1, missing 1
  [PASSED] edge: n=1, missing 0

  5/5 tests passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(missingNumber)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Sorting + linear scan | O(n log n) | O(1) |
| Hash set lookup | O(n) | O(n) |
| Gauss sum formula | O(n) | O(1) |
| XOR index-value pairs | O(n) | O(1) |

## Real World Connection

In Citi ETL pipelines, sequence numbers are assigned to batches
of records streaming from 6,000 endpoints. A missing sequence
number signals a dropped message — exactly the "missing number"
problem. The Gauss-sum approach lets an AWS Lambda validator
detect gaps in O(n) with zero extra memory, critical when
processing millions of events per hour in a serverless pipeline.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra